# Notebook 18 — asyncio queues & backpressure

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `17-pytest-fixtures-parametrize.ipynb`

**Next up:** `../python-foundations-beginner-to-advanced.ipynb` + `../CURRICULUM-A-Z.md`

---

## Learning objectives

- Connect producers and consumers with **`asyncio.Queue`**.
- Use **`maxsize`** for bounded backpressure before GPUs/APIs overload.
- Shut pipelines down with sentinel values + **`task_done`** discipline.

## Table of contents

1. **`Queue` basics**
2. **Bounded producers**
3. **Sentinel shutdown**
4. **Progressive drills — FIFO → bounded fan-out → cooperative shutdown**
5. **Exercise — `fan_in_merge`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · `asyncio.Queue` basics

*Explanation:* Tasks **`await queue.put`** when full and **`await queue.get`** when empty — matches ingestion workers draining embeddings.


In [ ]:
import asyncio

async def fifo_demo():
    q: asyncio.Queue[str] = asyncio.Queue()
    await q.put("chunk-a")
    await q.put("chunk-b")
    print(await q.get(), await q.get())

await fifo_demo()


## 2 · Bounded backpressure

*Explanation:* **`maxsize=N`** blocks aggressive producers until consumers catch up — protects RAM during burst uploads.


In [ ]:
import asyncio

async def bounded_demo():
    q: asyncio.Queue[int] = asyncio.Queue(maxsize=2)

    async def producer():
        for i in range(4):
            await q.put(i)
            print("produced", i)

    async def consumer():
        await asyncio.sleep(0.05)
        for _ in range(4):
            print("consumed", await q.get())

    await asyncio.gather(producer(), consumer())

await bounded_demo()


## 3 · Sentinel shutdown

*Explanation:* Push a **`None`** (or sentinel singleton) after real items so consumers exit cleanly.


In [ ]:
import asyncio

_STOP = object()


async def drain(q: asyncio.Queue):
    buf = []
    while True:
        item = await q.get()
        if item is _STOP:
            q.task_done()
            break
        buf.append(item)
        q.task_done()
    return buf


async def sentinel_demo():
    q: asyncio.Queue = asyncio.Queue()
    await q.put("a")
    await q.put(_STOP)
    print(await drain(q))

await sentinel_demo()


---

## Progressive drills — **easy → harder**

**Streaming ingestion** stacks queues between HTTP receivers and embedding workers — drills rehearse bounded FIFO semantics.


### A · Easiest — `join()` waits on producers

Call **`await queue.join()`** after scheduling producers when each item invokes **`task_done`**.


In [ ]:
import asyncio

async def join_demo():
    q: asyncio.Queue[int] = asyncio.Queue()

    async def worker():
        while True:
            item = await q.get()
            if item is None:
                q.task_done()
                break
            q.task_done()

    task = asyncio.create_task(worker())
    for val in (1, 2, 3):
        await q.put(val)
    await q.put(None)
    await q.join()
    await task

await join_demo()
print("join OK")


### B · Medium — deterministic multi-queue reads

Alternate **`get`** calls when each shard exposes ordered chunks.


In [ ]:
import asyncio

async def merge_demo():
    q1: asyncio.Queue[int] = asyncio.Queue()
    q2: asyncio.Queue[int] = asyncio.Queue()
    await q1.put(3)
    await q2.put(1)
    await q1.put(4)
    await q2.put(2)
    seq = [
        await q1.get(),
        await q2.get(),
        await q1.get(),
        await q2.get(),
    ]
    print(sorted(seq))

await merge_demo()


### C · Harder — paced producer vs consumer

`asyncio.sleep` inside producers simulates token refill pacing.


In [ ]:
import asyncio

async def paced_demo():
    q: asyncio.Queue[str] = asyncio.Queue(maxsize=2)

    async def producer():
        for lab in ["a", "b", "c"]:
            await asyncio.sleep(0.02)
            await q.put(lab)

    async def consumer():
        out = []
        for _ in range(3):
            out.append(await q.get())
        print("drained", out)

    await asyncio.gather(producer(), consumer())

await paced_demo()


### Exercise — `fan_in_merge`

Implement **`async def fan_in_merge(qs: list[asyncio.Queue[str]], stop_sentinel: object) -> list[str]`** returning **every non-sentinel string**, sorted alphabetically.

Rules:

- Initially schedule **`get()`** on **each** queue ( **`asyncio.create_task`** ).
- Whenever a **`get`** completes: if the value **`is`** **`stop_sentinel`**, **stop scheduling that queue**; otherwise append the string and schedule **`get()`** again on **that same queue**.
- When **no tasks remain**, return **`sorted(collected)`**.

Seed example:

```python
sentinel = object()
q1.put_nowait("delta"); q1.put_nowait(sentinel)
q2.put_nowait("alpha"); q2.put_nowait(sentinel)
```

Expect **`["alpha", "delta"]`**.


In [ ]:
import asyncio


async def fan_in_merge(qs: list[asyncio.Queue[str]], stop_sentinel: object) -> list[str]:
    raise NotImplementedError


async def exercise_main():
    sentinel = object()
    q1: asyncio.Queue[str] = asyncio.Queue()
    q2: asyncio.Queue[str] = asyncio.Queue()
    q1.put_nowait("delta")
    q1.put_nowait(sentinel)
    q2.put_nowait("alpha")
    q2.put_nowait(sentinel)
    got = await fan_in_merge([q1, q2], sentinel)
    assert got == ["alpha", "delta"]


await exercise_main()
print("OK")


### Solution — fan_in_merge

<details>
<summary>Click to expand</summary>

```python
import asyncio

async def fan_in_merge(qs: list[asyncio.Queue[str]], stop_sentinel: object) -> list[str]:
    collected: list[str] = []
    pending: dict[asyncio.Task[str], asyncio.Queue[str]] = {}

    def launch(q: asyncio.Queue[str]) -> None:
        pending[asyncio.create_task(q.get())] = q

    for q in qs:
        launch(q)

    while pending:
        done, _ = await asyncio.wait(pending.keys(), return_when=asyncio.FIRST_COMPLETED)
        task = done.pop()
        q = pending.pop(task)
        item = task.result()
        if item is stop_sentinel:
            continue
        collected.append(item)
        launch(q)

    return sorted(collected)
```

</details>
